# Stage 1 — AMZN Data Ingestion

Builds the four base artifacts every later stage consumes from real sources.
Stage 1 is the only stage that touches the network — once it has run successfully,
the rest of the pipeline operates entirely from `./artifacts/`.

## Sources
1. **FNSPID** (HuggingFace dataset `Zihan1004/FNSPID`) — AMZN news headlines.
2. **Yahoo Finance** via `yfinance` (with FNSPID full-history ZIP as fallback) — daily OHLCV.
3. **SEC EDGAR** XBRL company concepts — filing-based fundamentals.
4. **Derived** — weak sentiment labels generated from forward returns.

## Outputs (written to `./artifacts/`)
- `text_df.parquet` — `(date, text, label)` weak-labeled headlines for Stage 3.
- `price_df.parquet` — daily OHLCV plus engineered return/volatility features.
- `edgar_df.parquet` — filing-level wide table of EDGAR concepts.
- `num_df.parquet` — Stage 4's numerical feature table (prices + fundamentals + risk_score target).

## Weak-labeling logic
Headlines have no ground-truth sentiment, so labels are generated by looking
at the next-day forward return for AMZN: positive return above a
volatility-scaled threshold becomes "positive", below the negative threshold
becomes "negative", in between is "neutral". The threshold scaling factor is
chosen on the **train slice only** (no look-ahead) by sweeping until each class
has at least 10% of the training rows.

## Key assumptions and caveats
- Single ticker: AMZN (CIK `0001018724`). Configured analysis window: 2010-2023.
- **FNSPID coverage for AMZN spans April 2020 – December 2023 only.** Pre-2020
  numerical rows therefore have no headline coverage; Stage 4's `+NLP` variant
  falls back to neutral 1/3 priors for those rows.
- Liabilities are reconstructed via the accounting identity
  `Liabilities = Assets - StockholdersEquity` (see Section 1.3 for why).
- Revenues are coalesced from two GAAP tags spanning the ASC-606 transition.
- The `rnd_intensity` feature was dropped (AMZN tags R&D under a company
  extension, not standard us-gaap).

## Reproducibility
- FNSPID dataset revision is printed in Section 1.1.
- EDGAR pull date is printed in Section 1.3.
- The chronological cutoff used by all downstream stages is computed and
  printed in Section 1.5.


In [1]:
# Load shared config, constants, and helper functions used across stages.
from common import *

# io and zipfile are used for the FNSPID fallback ZIP parsing path.
import io
import zipfile
# date is only used to log when EDGAR pull was executed.
from datetime import date

# yfinance provides AMZN daily OHLCV data without an API key.
import yfinance as yf
# Hub utilities for locating/downloading fallback price ZIP files.
from huggingface_hub import hf_hub_download, list_repo_files

[common] device=cpu  artifacts=/cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts  results=/cluster/tufts/hrilab/jmonta04/modular_pipeline/results


## 1.1 FNSPID headlines for AMZN

In [2]:
def choose_column_by_keywords(columns, keyword_groups):
    """Return first column whose name contains all terms in a keyword group."""
    # Compare in lowercase so matching is robust to capitalization.
    normalized = {col: col.lower() for col in columns}

    # Try each candidate keyword pattern in order.
    for keywords in keyword_groups:
        for col, lower_col in normalized.items():
            # all(...) means every term in the pattern must appear.
            if all(token in lower_col for token in keywords):
                return col
    return None


def build_text_column(df, headline_col, body_col=None):
    """Create final text column from headline (+ optional article body)."""
    # Headline is the base text field used for modeling.
    text = df[headline_col].fillna("").astype(str).str.strip()

    # If a body/content column exists, append it for richer context.
    if body_col and body_col in df.columns:
        body = df[body_col].fillna("").astype(str).str.strip()
        has_body = body.str.len() > 0
        text = np.where(has_body, text + "\n\n" + body, text)
        text = pd.Series(text, index=df.index)

    return text


# Download raw FNSPID news CSV directly to avoid mixed-type Arrow inference errors.
news_csv_path = hf_hub_download(
    repo_id="Zihan1004/FNSPID",
    filename="Stock_news/nasdaq_exteral_data.csv",
    repo_type="dataset",
)
print(f"FNSPID news CSV local path: {news_csv_path}")

# Stream CSV in chunks and keep only AMZN rows before concatenation.
amzn_chunks = []
first_chunk_columns = None
symbol_col_candidates = [["stock", "symbol"], ["symbol"], ["ticker"]]

for chunk in pd.read_csv(news_csv_path, chunksize=500_000, low_memory=False):
    if first_chunk_columns is None:
        first_chunk_columns = list(chunk.columns)

    chunk_symbol_col = choose_column_by_keywords(
        columns=list(chunk.columns),
        keyword_groups=symbol_col_candidates,
    )
    if chunk_symbol_col is None:
        raise ValueError(
            "Could not infer symbol column while streaming FNSPID CSV. "
            f"Available columns: {list(chunk.columns)}"
        )

    amzn_chunk = chunk[
        chunk[chunk_symbol_col].astype(str).str.upper() == CONFIG["company"]["ticker"]
    ]
    if not amzn_chunk.empty:
        amzn_chunks.append(amzn_chunk)

if amzn_chunks:
    news_df = pd.concat(amzn_chunks, ignore_index=True)
elif first_chunk_columns is not None:
    news_df = pd.DataFrame(columns=first_chunk_columns)
else:
    raise RuntimeError("FNSPID CSV streaming produced no chunks.")

# Detect symbol/headline/date columns dynamically to handle schema drift.
symbol_col = choose_column_by_keywords(
    columns=list(news_df.columns),
    keyword_groups=[
        ["symbol"],
        ["ticker"],
        ["stock", "symbol"],
        ["stock", "ticker"],
    ],
)
headline_col = choose_column_by_keywords(
    columns=list(news_df.columns),
    keyword_groups=[
        ["headline"],
        ["title"],
        ["news", "title"],
        ["text"],
    ],
)
date_col = choose_column_by_keywords(
    columns=list(news_df.columns),
    keyword_groups=[
        ["published"],
        ["datetime"],
        ["timestamp"],
        ["date"],
        ["time"],
    ],
)
# Optional long-form text field.
body_col = choose_column_by_keywords(
    columns=list(news_df.columns),
    keyword_groups=[["body"], ["content"], ["article"], ["summary"]],
)

# Validate required fields before continuing.
required = {
    "symbol_col": symbol_col,
    "headline_col": headline_col,
    "date_col": date_col,
}
missing = [name for name, value in required.items() if value is None]
if missing:
    raise ValueError(
        "Could not infer required FNSPID columns. "
        f"Missing: {missing}. Available columns: {list(news_df.columns)}"
    )

print("Detected FNSPID columns:")
print(f"  symbol:   {symbol_col}")
print(f"  headline: {headline_col}")
print(f"  date:     {date_col}")
print(f"  body:     {body_col}")

# Keep only AMZN rows based on ticker/symbol match.
amzn_news = news_df[news_df[symbol_col].astype(str).str.upper() == CONFIG["company"]["ticker"]].copy()

# Parse timestamps in UTC first, then drop timezone and keep date only.
parsed_dates = pd.to_datetime(amzn_news[date_col], errors="coerce", utc=True)
amzn_news["date"] = parsed_dates.dt.tz_localize(None).dt.normalize()

# Build final training text from headline and optional body.
amzn_news["text"] = build_text_column(amzn_news, headline_col=headline_col, body_col=body_col)

# Keep only required columns for downstream NLP stages.
text_df = amzn_news[["date", "text"]].copy()
text_df = text_df.dropna(subset=["date", "text"])
text_df = text_df[text_df["text"].str.strip().str.len() > 0]

# Restrict to configured analysis window.
window_start = pd.Timestamp(CONFIG["company"]["start_date"])
window_end = pd.Timestamp(CONFIG["company"]["end_date"])
text_df = text_df[(text_df["date"] >= window_start) & (text_df["date"] <= window_end)]

# Sort by date so chronology is explicit for all later stages.
text_df = text_df.sort_values("date").reset_index(drop=True)

# Print data coverage summary.
print(f"AMZN headline rows: {len(text_df)}")
print(f"Date range: {text_df['date'].min()} -> {text_df['date'].max()}")

# year_hist is a per-year volume check for headline coverage.
year_hist = text_df.assign(year=text_df["date"].dt.year).groupby("year").size()
print("Headlines per year:")
print(year_hist.to_string())

# Preview cleaned text data.
text_df.head()

FNSPID news CSV local path: /cluster/tufts/hrilab/jmonta04/.hf_cache/hub/datasets--Zihan1004--FNSPID/snapshots/bf9189c41527198897d1af3e17b1a0095279fc45/Stock_news/nasdaq_exteral_data.csv


Detected FNSPID columns:
  symbol:   Stock_symbol
  headline: Article_title
  date:     Date
  body:     Article_title


AMZN headline rows: 10521
Date range: 2009-06-22 00:00:00 -> 2023-12-16 00:00:00
Headlines per year:
year
2009      40
2010     348
2011     324
2012     489
2013     554
2014     947
2015     786
2016     914
2017     883
2018     783
2019     805
2020     831
2021     556
2022     966
2023    1295


,date,text
0,2009-06-22,guest post - Best Buy's been a 'best buy'\n\ng...
1,2009-08-10,AnalystChoice.com Brings You the Best Complime...
2,2009-08-16,Time to Refill on Coca-Cola Shares - Barron's\...
3,2009-09-09,Zignals Stock Charts: 7 Divided Elites from Th...
4,2009-09-18,IBM (IBM) and Coca-Cola (KO) Retains Top Brand...


## 1.2 Daily AMZN price data (Yahoo Finance with FNSPID fallback)

In [3]:
def compute_price_features(price_df):
    """Add return/volatility features derived from daily close prices."""
    df = price_df.copy()

    # Simple historical returns over 1, 5, and 21 trading days.
    df["ret_1d"] = df["close"].pct_change(1)
    df["ret_5d"] = df["close"].pct_change(5)
    df["ret_21d"] = df["close"].pct_change(21)

    # Rolling volatility estimates from daily returns.
    df["vol_21d"] = df["ret_1d"].rolling(21).std()
    df["vol_63d"] = df["ret_1d"].rolling(63).std()

    # Drawdown measures distance from trailing 252-day high.
    df["drawdown"] = (df["close"] / df["close"].rolling(252).max()) - 1.0
    return df


def load_prices_from_yfinance(ticker, start_date, end_date):
    """Pull daily OHLCV from Yahoo Finance for configured ticker/date range."""
    history = yf.download(
        tickers=ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=False,
        progress=False,
    )
    # Return None so caller can trigger fallback path.
    if history.empty:
        return None

    # yfinance >=0.2.40 returns a 2-level column index (price field, ticker) even when
    # only a single ticker is downloaded. We drop the ticker level so downstream code
    # can do df["close"] and get a Series; without this flatten, df["close"] would be
    # a 1-column DataFrame and pct_change/rolling chains would silently misbehave.
    if isinstance(history.columns, pd.MultiIndex):
        history.columns = history.columns.get_level_values(0)

    # Flatten index and standardize expected column names.
    history = history.reset_index()
    history = history.rename(
        columns={
            "Date": "date",
            "Close": "close",
            "Volume": "volume",
            "Open": "open",
            "High": "high",
            "Low": "low",
        }
    )

    # Keep date-only timestamps and required columns.
    history["date"] = pd.to_datetime(history["date"]).dt.normalize()
    keep_cols = ["date", "open", "high", "low", "close", "volume"]
    return history[keep_cols].copy()


def load_prices_from_fnspid_fallback(ticker, start_date, end_date):
    """Fallback source: parse FNSPID full-history ZIP if yfinance fails."""
    repo_id = "Zihan1004/FNSPID"

    # Discover ZIP files in repo that look like full history dumps.
    files = list_repo_files(repo_id=repo_id, repo_type="dataset")
    zip_candidates = [path for path in files if "full_history" in path and path.endswith(".zip")]
    if not zip_candidates:
        raise FileNotFoundError("FNSPID fallback ZIP file not found in dataset repo.")

    # Download the first candidate ZIP.
    zip_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=zip_candidates[0],
    )

    with zipfile.ZipFile(zip_path, "r") as archive:
        # Look for CSV members inside the ZIP.
        csv_members = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if not csv_members:
            raise FileNotFoundError("No CSV file found inside FNSPID fallback ZIP.")

        fallback_df = None
        for member in csv_members:
            with archive.open(member, "r") as raw_file:
                candidate = pd.read_csv(io.BytesIO(raw_file.read()))

            # Flexible column mapping in case schema differs across files.
            lower_columns = {col.lower(): col for col in candidate.columns}
            symbol_col = lower_columns.get("symbol") or lower_columns.get("ticker")
            date_col = lower_columns.get("date") or lower_columns.get("datetime")
            close_col = lower_columns.get("close")
            volume_col = lower_columns.get("volume")

            if symbol_col and date_col and close_col:
                # Keep only AMZN rows.
                ticker_rows = candidate[candidate[symbol_col].astype(str).str.upper() == ticker].copy()
                if not ticker_rows.empty:
                    # Parse dates and restrict to configured time window.
                    ticker_rows["date"] = pd.to_datetime(ticker_rows[date_col], errors="coerce").dt.normalize()
                    ticker_rows = ticker_rows[(ticker_rows["date"] >= pd.Timestamp(start_date)) & (ticker_rows["date"] <= pd.Timestamp(end_date))]

                    # Convert numeric fields safely; invalid values become NaN.
                    ticker_rows["close"] = pd.to_numeric(ticker_rows[close_col], errors="coerce")
                    ticker_rows["volume"] = pd.to_numeric(ticker_rows[volume_col], errors="coerce") if volume_col else np.nan

                    # Fallback files may not include full OHLC; fill missing fields with NaN.
                    ticker_rows["open"] = np.nan
                    ticker_rows["high"] = np.nan
                    ticker_rows["low"] = np.nan

                    fallback_df = ticker_rows[["date", "open", "high", "low", "close", "volume"]].copy()
                    break

    if fallback_df is None or fallback_df.empty:
        raise RuntimeError("FNSPID fallback price extraction did not return AMZN rows.")

    return fallback_df.sort_values("date").reset_index(drop=True)


# Pull ticker/date settings from shared config.
ticker = CONFIG["company"]["ticker"]
start_date = CONFIG["company"]["start_date"]
end_date = CONFIG["company"]["end_date"]

# Primary source: Yahoo Finance.
price_df = load_prices_from_yfinance(ticker=ticker, start_date=start_date, end_date=end_date)
if price_df is None:
    print("yfinance returned no rows. Falling back to FNSPID full history ZIP...")
    # Backup source: FNSPID ZIP.
    price_df = load_prices_from_fnspid_fallback(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date,
    )

# Ensure chronological order, then add derived features.
price_df = price_df.sort_values("date").reset_index(drop=True)
price_df = compute_price_features(price_df)

# Print coverage and preview.
print(f"Price rows: {len(price_df)}")
print(f"Price date range: {price_df['date'].min()} -> {price_df['date'].max()}")
price_df.head()

Price rows: 3774
Price date range: 2009-01-02 00:00:00 -> 2023-12-29 00:00:00


Price,date,open,high,low,close,volume,ret_1d,ret_5d,ret_21d,vol_21d,vol_63d,drawdown
0,2009-01-02,22.700001,23.000000,22.520000,22.950001,16355800,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-01-05,22.785000,22.945000,22.590000,22.719999,20237200,-0.010022,NaN,NaN,NaN,NaN,NaN
2,2009-01-06,22.850000,22.985001,22.230000,22.355000,21307800,-0.016065,NaN,NaN,NaN,NaN,NaN
3,2009-01-07,22.110001,22.590000,22.110001,22.465000,21581000,0.004921,NaN,NaN,NaN,NaN,NaN
4,2009-01-08,22.420000,22.639999,22.280001,22.620001,20087000,0.006900,NaN,NaN,NaN,NaN,NaN


## 1.3–1.5 EDGAR fundamentals, numerical table, weak labels, and persistence

In [4]:
def extract_edgar_observations(concept_payload, concept_name):
    """Parse one concept payload into filing-level rows we can tabulate."""
    # EDGAR stores values under a unit key (USD, shares, pure, ...).
    units = concept_payload.get("units", {})
    preferred_order = ["USD", "shares", "pure"]

    # Try preferred units first, then fall back to first available unit.
    unit_name = None
    for candidate in preferred_order:
        if candidate in units:
            unit_name = candidate
            break
    if unit_name is None and units:
        unit_name = next(iter(units.keys()))

    if unit_name is None:
        return pd.DataFrame()

    rows = []
    for item in units.get(unit_name, []):
        # Keep only annual/quarterly filing forms.
        form_type = item.get("form")
        if form_type not in {"10-K", "10-Q"}:
            continue

        # Build normalized row fields used downstream.
        rows.append(
            {
                "concept": concept_name,
                "unit": unit_name,
                "period_end": pd.to_datetime(item.get("end"), errors="coerce"),
                "filed": pd.to_datetime(item.get("filed"), errors="coerce"),
                "val": pd.to_numeric(item.get("val"), errors="coerce"),
                "fp": item.get("fp"),
                "fy": item.get("fy"),
                "form": form_type,
            }
        )

    if not rows:
        return pd.DataFrame()

    obs = pd.DataFrame(rows)
    obs = obs.dropna(subset=["period_end", "filed", "val"])
    obs["period_end"] = obs["period_end"].dt.normalize()
    obs["filed"] = obs["filed"].dt.normalize()

    # Restatements can produce duplicate period_end entries.
    # Keep the newest filing for each (period_end, concept).
    obs = obs.sort_values(["period_end", "filed"])
    obs = obs.drop_duplicates(subset=["period_end", "concept"], keep="last")
    return obs


def build_edgar_df(cik, concepts):
    """Download and pivot all requested EDGAR concepts into one wide table."""
    all_rows = []
    for concept in concepts:
        # Pull concept JSON via cached helper in common.py.
        try:
            payload = get_company_concept(cik=cik, concept=concept)
        except requests.exceptions.HTTPError as exc:
            if exc.response is not None and exc.response.status_code == 404:
                print(f"Concept not filed by this company (skipping): {concept}")
                continue
            raise
        concept_rows = extract_edgar_observations(payload, concept_name=concept)
        if concept_rows.empty:
            print(f"No filing rows for concept: {concept}")
            continue
        all_rows.append(concept_rows)

    if not all_rows:
        raise RuntimeError("No EDGAR concept rows were collected.")

    # Long format = one row per concept per filing.
    long_df = pd.concat(all_rows, ignore_index=True)
    long_df = long_df.sort_values(["filed", "period_end", "concept"])

    # Wide format = one row per filing date with concept columns.
    wide_df = long_df.pivot_table(
        index=["filed", "period_end", "form", "fy", "fp"],
        columns="concept",
        values="val",
        aggfunc="last",
    ).reset_index()

    # Carry forward values so each filing state remains available until next filing.
    wide_df = wide_df.sort_values("filed").reset_index(drop=True)
    concept_cols = [col for col in wide_df.columns if col in concepts]
    wide_df[concept_cols] = wide_df[concept_cols].ffill()
    return wide_df


def compute_forward_realized_volatility(ret_1d, horizon=21):
    """Compute forward (future) realized volatility over next N days."""
    values = ret_1d.to_numpy(dtype=float)
    output = np.full(len(values), np.nan)

    for idx in range(len(values)):
        # Window starts tomorrow to keep target fully forward-looking.
        start = idx + 1
        end = idx + 1 + horizon
        if end <= len(values):
            window = values[start:end]
            if np.isfinite(window).all():
                output[idx] = float(np.std(window, ddof=1))

    return pd.Series(output, index=ret_1d.index)


def minmax_scale(series):
    """Scale series into [0, 1] range; return NaN if scale is undefined."""
    minimum = series.min()
    maximum = series.max()
    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(np.nan, index=series.index)
    return (series - minimum) / (maximum - minimum)


def build_num_df(price_df, edgar_df):
    """Create Stage 4 numerical table from prices + point-in-time EDGAR facts."""
    # As-of join by filed date prevents leakage from future filings.
    merged = pd.merge_asof(
        left=price_df.sort_values("date"),
        right=edgar_df.sort_values("filed"),
        left_on="date",
        right_on="filed",
        direction="backward",
    )

    # KO revenue can arrive under two tags depending on filing period. Coalesce
    # modern `Revenues` with the older `SalesRevenueGoodsNet` bridge so one
    # continuous Revenues series is available for ratio features.
    if (
        "Revenues" in merged.columns
        or "SalesRevenueGoodsNet" in merged.columns
    ):
        rev_modern = merged.get("Revenues")
        rev_legacy = merged.get("SalesRevenueGoodsNet")
        if rev_modern is not None and rev_legacy is not None:
            merged["Revenues"] = rev_modern.combine_first(rev_legacy)
        elif rev_modern is not None:
            merged["Revenues"] = rev_modern
        else:
            merged["Revenues"] = rev_legacy

    # Reconstruct total liabilities via the accounting identity:
    #     Liabilities = Assets - StockholdersEquity
    # KO, like AMZN, does not provide a clean single `us-gaap:Liabilities` series
    # for this workflow. Using the accounting identity keeps liabilities consistent
    # from two reliably populated concepts (Assets and StockholdersEquity).
    merged["Liabilities"] = merged["Assets"] - merged["StockholdersEquity"]

    # Fundamental ratios and scale features.
    merged["debt_to_equity"] = merged["LongTermDebt"] / merged["StockholdersEquity"]
    merged["current_ratio"] = merged["Assets"] / merged["Liabilities"]
    merged["profit_margin"] = merged["NetIncomeLoss"] / merged["Revenues"]
    merged["roe"] = merged["NetIncomeLoss"] / merged["StockholdersEquity"]
    merged["log_assets"] = np.log(merged["Assets"].clip(lower=1.0))

    # Define risk target as future realized volatility over 21 trading days.
    merged["realized_vol_21d_fwd"] = compute_forward_realized_volatility(
        merged["ret_1d"],
        horizon=21,
    )
    merged["risk_score"] = minmax_scale(merged["realized_vol_21d_fwd"])

    # Keep only features used by Stage 4 models.
    feature_cols = [
        "date",
        "ret_1d",
        "ret_5d",
        "ret_21d",
        "vol_21d",
        "vol_63d",
        "drawdown",
        "debt_to_equity",
        "current_ratio",
        "profit_margin",
        "roe",
        "log_assets",
        "risk_score",
    ]
    num_df = merged[feature_cols].copy()

    # Clean invalid numeric values before model training.
    num_df = num_df.replace([np.inf, -np.inf], np.nan)
    num_df = num_df.dropna().reset_index(drop=True)
    return num_df


def compute_headline_weak_labels(
    raw_text_df,
    prices_df,
    train_end_date,
    horizon_days=1,
):
    """Assign weak sentiment labels from forward returns with vol scaling."""
    labeled = raw_text_df.copy()
    labeled["date"] = pd.to_datetime(labeled["date"]).dt.normalize()

    # Keep pricing fields required for forward-return labeling.
    prices = prices_df[["date", "close", "vol_21d"]].dropna().copy()
    prices = prices.sort_values("date").reset_index(drop=True)
    trading_dates = prices["date"].to_numpy()

    # Map each headline date to the next trading day index.
    next_index = np.searchsorted(trading_dates, labeled["date"].to_numpy(), side="left")

    # Preallocate arrays for speed and explicitness.
    forward_returns = np.full(len(labeled), np.nan)
    daily_vol = np.full(len(labeled), np.nan)

    for row_idx, start_idx in enumerate(next_index):
        end_idx = start_idx + horizon_days
        if start_idx < len(prices) and end_idx < len(prices):
            start_close = prices.iloc[start_idx]["close"]
            end_close = prices.iloc[end_idx]["close"]

            # Forward return over configured horizon.
            forward_returns[row_idx] = (end_close / start_close) - 1.0
            # Daily volatility used to scale decision threshold.
            daily_vol[row_idx] = prices.iloc[start_idx]["vol_21d"]

    labeled["forward_return"] = forward_returns
    labeled["daily_vol"] = daily_vol
    labeled = labeled.dropna(subset=["forward_return", "daily_vol"]).reset_index(drop=True)

    train_end_date = pd.to_datetime(train_end_date).normalize()
    train_mask = labeled["date"] <= train_end_date
    train_slice = labeled[train_mask]

    if train_slice.empty:
        raise ValueError(
            "Weak-label threshold sweep failed: training slice is empty. "
            "Check train_end_date and upstream date coverage."
        )

    # Weak-label threshold sweep.
    #
    # We don't have labeled sentiment for these headlines, so we generate weak labels
    # from the next-day forward return: positive label if return > +factor * daily_vol,
    # negative if return < -factor * daily_vol, neutral otherwise. The factor scales
    # the threshold to the volatility regime so labels stay meaningful as vol changes.
    #
    # Three deliberate choices in this loop:
    #   1. We sweep on the TRAINING SLICE ONLY (rows up to train_end_date). Choosing
    #      a threshold based on test-set returns would leak future information into
    #      the labels and inflate downstream Stage 3 metrics.
    #   2. We require each class to have >= 10% share so neither class collapses to
    #      a degenerate fraction; otherwise the classifier could trivially "win" by
    #      always predicting the dominant class.
    #   3. We pick the smallest factor that satisfies the >=10% constraint, biasing
    #      toward more informative (less neutral) labels when possible.
    chosen_factor = None
    final_shares = None
    for factor in np.arange(0.5, 2.01, 0.1):
        upper = factor * train_slice["daily_vol"]
        lower = -factor * train_slice["daily_vol"]
        train_labels = np.where(
            train_slice["forward_return"] > upper,
            "positive",
            np.where(train_slice["forward_return"] < lower, "negative", "neutral"),
        )
        shares = pd.Series(train_labels).value_counts(normalize=True)
        min_share = shares.reindex(["negative", "neutral", "positive"]).fillna(0.0).min()
        final_shares = shares
        if min_share >= 0.10:
            chosen_factor = factor
            break

    if chosen_factor is None:
        raise ValueError(
            "Weak-label threshold sweep failed: no factor in [0.5, 2.0] produced "
            "≥10% share in every class on the training slice. "
            f"Final shares at factor=2.0: {dict(final_shares)}. "
            "Extend the sweep range or revisit the labeling rule."
        )

    # Final labels using selected threshold factor.
    upper = chosen_factor * labeled["daily_vol"]
    lower = -chosen_factor * labeled["daily_vol"]
    label_text = np.where(
        labeled["forward_return"] > upper,
        "positive",
        np.where(labeled["forward_return"] < lower, "negative", "neutral"),
    )

    labeled["label_text"] = label_text
    labeled["label"] = labeled["label_text"].map(LABEL_TO_ID)

    train_counts = labeled.loc[train_mask, "label_text"].value_counts()
    test_counts = labeled.loc[~train_mask, "label_text"].value_counts()

    print(
        f"Weak label threshold factor chosen: {chosen_factor:.2f} "
        "(selected on train slice only)"
    )
    print("Train class counts:")
    print(train_counts.reindex(["negative", "neutral", "positive"]).fillna(0).astype(int).to_string())
    print("Test class counts:")
    print(test_counts.reindex(["negative", "neutral", "positive"]).fillna(0).astype(int).to_string())

    # Return Stage 3-ready columns only.
    return labeled[["date", "text", "label"]].reset_index(drop=True)


# Track EDGAR pull execution date for reproducibility.
edgar_pull_date = date.today().isoformat()
print(f"EDGAR pull date: {edgar_pull_date}")

# Pull, parse, and pivot EDGAR concepts.
edgar_df = build_edgar_df(
    cik=CONFIG["company"]["cik"],
    concepts=CONFIG["edgar"]["concepts"],
)

# Quick quality check at both ends of the filing timeline.
print("EDGAR first 5 rows:")
print(edgar_df.head().to_string())
print("\nEDGAR last 5 rows:")
print(edgar_df.tail().to_string())

# Build numerical model table and weak-labeled text table.
num_df = build_num_df(price_df=price_df, edgar_df=edgar_df)
train_end_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)
print(f"Shared chronological cutoff: {train_end_date.date()}")

text_df = compute_headline_weak_labels(
    raw_text_df=text_df,
    prices_df=price_df,
    train_end_date=train_end_date,
    horizon_days=CONFIG["labeling"]["horizon_days"],
)

# Persist artifacts for downstream stages.
save_text_df(text_df)
save_price_df(price_df)
save_edgar_df(edgar_df)
save_num_df(num_df)

print("Saved artifacts:")
print(f"  {ARTIFACTS_DIR / 'text_df.parquet'}")
print(f"  {ARTIFACTS_DIR / 'price_df.parquet'}")
print(f"  {ARTIFACTS_DIR / 'edgar_df.parquet'}")
print(f"  {ARTIFACTS_DIR / 'num_df.parquet'}")
print(f"  EDGAR cache dir: {EDGAR_CACHE_DIR}")

EDGAR pull date: 2026-05-07


EDGAR first 5 rows:
concept      filed period_end  form    fy  fp        Assets  CashAndCashEquivalentsAtCarryingValue  EarningsPerShareBasic  LongTermDebt  NetIncomeLoss  OperatingIncomeLoss  Revenues  SalesRevenueGoodsNet  StockholdersEquity
0       2009-07-30 2008-06-27  10-Q  2009  Q2           NaN                           6.571000e+09                   0.61           NaN   1.422000e+09         2.679000e+09       NaN          9.046000e+09                 NaN
1       2009-07-30 2009-07-03  10-Q  2009  Q2  4.605400e+10                           6.571000e+09                   0.88           NaN   1.422000e+09         2.679000e+09       NaN          9.046000e+09        2.307600e+10
2       2010-02-26 2006-12-31  10-K  2009  FY  4.605400e+10                           2.440000e+09                   0.88           NaN   1.422000e+09         2.679000e+09       NaN          9.046000e+09        2.307600e+10
3       2010-02-26 2007-12-31  10-K  2009  FY  4.605400e+10                         

Weak label threshold factor chosen: 0.50 (selected on train slice only)
Train class counts:
label_text
negative    2213
neutral     3410
positive    2352
Test class counts:
label_text
negative     712
neutral     1111
positive     723
Saved artifacts:
  /cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts/text_df.parquet
  /cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts/price_df.parquet
  /cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts/edgar_df.parquet
  /cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts/num_df.parquet
  EDGAR cache dir: /cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts/edgar_cache
